This app randomly generates a male name (first name and surname) from a list of names. The names are all male because I intend to use them as names of baseball prospects later in my baseball sim project.

In [ ]:
import random

We define a dictionary `profile` that contains a list of all the names we wish to include and their corresponding weights. The probability that a name is chosen is `weight/sum(weight)`. For example, if we have `{"David": 4, "John": 6, "Miguel": 4, "Anthony": 5}`, then

$$ P(\textrm{David}) = \frac{10}{10+6+4+5} = 0.4 = 40\%. $$

In [ ]:
profile = {
    "first_names": [
        {"name": "David", "weight": 10},
        {"name": "John", "weight": 6},
        {"name": "Miguel", "weight": 4},
        {"name": "Anthony", "weight": 5}
    ],
    "last_names": [
        {"name": "Ross", "weight": 5},
        {"name": "Martinez", "weight": 8},
        {"name": "Sullivan", "weight": 3},
        {"name": "Parker", "weight": 4}
    ]
}

The helper function `weighted_pick` selects a name from the first name or surname list, using the weights to calculate the probability, while `generate_name` combines the chosen first and last names together to produce a final name string.

In [ ]:
def weighted_pick(entries):
    names = [entry["name"] for entry in entries]
    weights = [entry["weight"] for entry in entries]
    return random.choices(names, weights=weights, k=1)[0]


def generate_name(profile):
    first = weighted_pick(profile["first_names"])
    last = weighted_pick(profile["last_names"])
    return f"{first} {last}"

In [ ]:
print(generate_name(profile))

David Parker


We can generate multiple names just by iterating `generate_name` using a loop.

In [ ]:
def generate_multiple(profile, n):
    names = []
    for _ in range(n):
        names.append(generate_name(profile))
    return "\n".join(names)


print(generate_multiple(profile, 10))

John Ross
Miguel Parker
David Martinez
John Parker
John Sullivan
David Sullivan
John Ross
John Martinez
Anthony Parker
Anthony Martinez


We call the dictionary containing the names a "profile" because one of our endgoals is to be able to randomly select from multiple lists of names loosely representing ethnicities. A "profile", in that context, represents a set of these lists, likely with its own probability scaffolding, designed to simulate some kind of cultural entity, like a nation, or a certain time period.

For example, the American "profile" would likely contain a list of English names, African-American names, Asian-American names, and Hispanic names, each within its own list category. A rough probability distribution may be 50% English, 25% Hispanic, 15% African-American, and 10% Asian, for example. Owing to America's history as a destination for immigrants, maybe 0.5% is taken off of each ethnicity and collected into a 2% Other.

An example of how this may be implemented in our Python generator is through the use of the new dictionary `profiles`:

In [ ]:
profiles = {
    "American": {
        "first_names": [
            {"name": "David", "weight": 10},
            {"name": "John", "weight": 6},
            {"name": "Anthony", "weight": 5}
        ],
        "last_names": [
            {"name": "Ross", "weight": 5},
            {"name": "Sullivan", "weight": 3},
            {"name": "Parker", "weight": 4}
        ]
    },
    "Dominican": {
        "first_names": [
            {"name": "José", "weight": 9},
            {"name": "Miguel", "weight": 7},
            {"name": "Juan", "weight": 6}
        ],
        "last_names": [
            {"name": "Martinez", "weight": 8},
            {"name": "Rodríguez", "weight": 7},
            {"name": "Soto", "weight": 5}
        ]
    }
}

In [ ]:
print(generate_multiple(profiles["American"], 5))
print()
print(generate_multiple(profiles["Dominican"], 5))

David Parker
David Sullivan
David Parker
John Sullivan
Anthony Parker

José Soto
José Rodríguez
Juan Soto
José Martinez
José Soto


More specifically, `profiles` is a dictionary where each key maps to a list of dictionaries. This structure, while a bit cumbersome from a human perspective, allows us to achieve modularity in our various functions (like `generate_multiple`) without the use of extra code.

If we instead want weighted probabilistic selection on the ethnicities themselves, however, we need to add an extra layer of list structure. As such, `profiles` will become a *list of dictionaries* where each key maps to a list of dictionaries. (Confusing, I know!)

In [ ]:
profiles = [
    {
        "name": "American",
        "weight": 70,
        "data": {
            "first_names": [
                {"name": "David", "weight": 10},
                {"name": "John", "weight": 6},
                {"name": "Anthony", "weight": 5}
            ],
            "last_names": [
                {"name": "Ross", "weight": 5},
                {"name": "Sullivan", "weight": 3},
                {"name": "Parker", "weight": 4}
            ]
        }
    },
    {
        "name": "Dominican",
        "weight": 30,
        "data": {
            "first_names": [
                {"name": "José", "weight": 9},
                {"name": "Miguel", "weight": 7},
                {"name": "Juan", "weight": 6}
            ],
            "last_names": [
                {"name": "Martinez", "weight": 8},
                {"name": "Rodríguez", "weight": 7},
                {"name": "Soto", "weight": 5}
            ]
        }
    }
]

The logic for profile section works similarly to the logic for name selection.

In [ ]:
def pick_profile(profiles):
    names = [p["name"] for p in profiles]
    weights = [p["weight"] for p in profiles]

    chosen_name = random.choices(names, weights=weights, k=1)[0]

    # find the matching profile
    for p in profiles:
        if p["name"] == chosen_name:
            return p

In [ ]:
def generate_name_from_profiles(profiles):
    profile = pick_profile(profiles)
    return generate_name(profile["data"])

In [ ]:
def generate_multiple_from_profiles(profiles, n):
    names = []
    for _ in range(n):
        names.append(generate_name_from_profiles(profiles))
    return "\n".join(names)

In [ ]:
print(generate_multiple_from_profiles(profiles, 10))

Anthony Ross
David Ross
John Ross
David Ross
David Sullivan
David Parker
Anthony Ross
Anthony Sullivan
Juan Martinez
Miguel Rodríguez


Under the default settings, 70% of the names are American and 30% are Dominican.

Now, we have a two-layer weighted generator: it picks the profile (with weighted probabilities), then it picks names within the profile. The actual profile selection requires a search on a manually provided name (like "American"), which is a bit inefficient, but it's good enough for now until we get a UI going.

Before we move on, we note that `generate_multiple_from_profiles` will pick a new profile for each name generated. As such, there is a mix of American and Dominican names in the list. If we want to instead generate all names in the list from a single static profile, we can use this function:

In [ ]:
def generate_multiple_from_one_profile(profiles, n):
    profile = pick_profile(profiles)

    names = []
    for _ in range(n):
        names.append(generate_name(profile["data"]))

    return "\n".join(names)

In [ ]:
print(generate_multiple_from_one_profile(profiles, 10))

David Parker
David Parker
Anthony Sullivan
David Sullivan
David Parker
David Parker
David Sullivan
John Sullivan
Anthony Parker
David Parker


This command randomly selects a profile, then generates ten names from that profile. "American" is selected 70% of the time, "Dominican" the rest.

We can do even better by combining both profile functionalities into a single function call with a `mode` argument. Selecting `mode="mixed"` will generate from the desired distribution of profiles selected while `mode="locked"` generates from only a single profile.

In [ ]:
def generate_batch(profiles, n, mode="mixed"):
    if mode == "mixed":
        return generate_multiple_from_profiles(profiles, n)

    elif mode == "locked":
        return generate_multiple_from_one_profile(profiles, n)

    else:
        raise ValueError("mode must be either 'mixed' or 'locked'")

In [ ]:
print(generate_batch(profiles, 10, mode="mixed"))

Anthony Parker
David Parker
José Soto
David Ross
John Parker
John Parker
Anthony Ross
David Ross
José Martinez
Miguel Martinez


In [ ]:
print(generate_batch(profiles, 10, mode="locked"))

John Ross
David Parker
David Ross
John Ross
Anthony Sullivan
John Parker
David Parker
David Parker
Anthony Parker
Anthony Parker


But what if we want to run the generator for a specific, named profile? Right now, `mode="locked"` will certainly lock a specific profile in place, but only after randomly selecting one via a given distribution. If we want to tell it very specifically generate, say, Dominican names, we have no direct way of doing this.

As such, we will write a `get_profile_by_name` helper first that retrieves a profile if it exists, then we will run an updated version of `generate_batch` that accounts for this new mode.

In [ ]:
def get_profile_by_name(profiles, profile_name):
    for profile in profiles:
        if profile["name"] == profile_name:
            return profile

    raise ValueError(f"No profile named {profile_name!r} was found.")

In [ ]:
def generate_batch(profiles, n, mode="mixed", profile_name=None):
    if mode == "mixed":
        return generate_multiple_from_profiles(profiles, n)

    elif mode == "locked":
        return generate_multiple_from_one_profile(profiles, n)

    elif mode == "specific":
        if profile_name is None:
            raise ValueError("profile_name is required when mode='specific'.")

        profile = get_profile_by_name(profiles, profile_name)

        names = []
        for _ in range(n):
            names.append(generate_name(profile["data"]))

        return "\n".join(names)

    else:
        raise ValueError("mode must be 'mixed', 'locked', or 'specific'.")

In [ ]:
print(generate_batch(profiles, 10, mode="specific", profile_name="Dominican"))

Miguel Rodríguez
Miguel Martinez
Juan Rodríguez
Juan Martinez
Miguel Rodríguez
José Martinez
Juan Soto
José Rodríguez
Juan Martinez
José Rodríguez


In [ ]:
print(generate_batch(profiles, 10, mode="specific", profile_name="American"))

John Sullivan
David Ross
David Ross
David Sullivan
Anthony Parker
Anthony Ross
John Parker
Anthony Parker
John Ross
John Sullivan


It is at this point where we would like to start moving the data to a more permanent location outside of Colab into, say, its own JSON file. This will prepare us for the ability to use a much larger data bank of names as desired, though we will need to handle a few other things first.

In [ ]:
import json

with open("profiles.json", "w", encoding="utf-8") as file:
    json.dump(profiles, file, indent=4, ensure_ascii=False)

The UTF-8 encoding is very important here because some names will have accented characters, such as in `José Rodríguez`.

In [ ]:
def load_profiles(filename):
    with open(filename, "r", encoding="utf-8") as file:
        return json.load(file)

In [ ]:
loaded_profiles = load_profiles("profiles.json")

print(generate_batch(loaded_profiles, 10, mode="mixed"))

José Soto
David Sullivan
David Ross
David Sullivan
Juan Rodríguez
Miguel Martinez
José Soto
David Parker
John Ross
Anthony Ross


At this point, we don't need hardcoded `profiles` at all. We can just run everything directly from an imported JSON file by defining `profiles` like so:

In [ ]:
profiles = load_profiles("profiles.json")

The following calls of our usual name-generating functions should all still work.

In [ ]:
print(generate_batch(profiles, 10, mode="mixed"))
print()
print(generate_batch(profiles, 10, mode="locked"))
print()
print(generate_batch(profiles, 10, mode="specific", profile_name="American"))

Anthony Parker
Anthony Sullivan
John Sullivan
David Parker
Miguel Rodríguez
Anthony Sullivan
David Ross
John Ross
Anthony Parker
Anthony Ross

John Ross
David Parker
John Sullivan
Anthony Sullivan
David Ross
David Parker
Anthony Parker
David Ross
John Parker
David Parker

David Parker
Anthony Ross
John Parker
David Ross
David Sullivan
John Ross
David Parker
David Ross
John Sullivan
Anthony Parker


We will proceed with a proper validation sweep later, but for now, this modification to `load_profiles` will at least ensure our code does not crash in a messy way if there's a missing or malformed JSON file.

In [ ]:
def load_profiles(filename):
    try:
        with open(filename, "r", encoding="utf-8") as file:
            return json.load(file)
    except FileNotFoundError:
        raise ValueError(f"File {filename!r} not found.")
    except json.JSONDecodeError:
        raise ValueError(f"File {filename!r} is not valid JSON.")

Now, suppose we wish to use a temporary one-time-only profile for the name distirbution instead of the default provided in the `profiles` JSON. For that, we need a temporary copy of `profiles` that we are free to overwrite or alter, which is where our next helper comes in:

In [ ]:
import copy

def make_custom_mix(profiles, weights_by_name):
    custom_profiles = copy.deepcopy(profiles)

    for profile in custom_profiles:
        if profile["name"] in weights_by_name:
            profile["weight"] = weights_by_name[profile["name"]]
        else:
            profile["weight"] = 0

    return custom_profiles

In [ ]:
custom_mix = make_custom_mix(
    profiles,
    {
        "American": 20,
        "Dominican": 80
    }
)

print(generate_batch(custom_mix, 20, mode="mixed"))

Miguel Soto
Anthony Parker
Miguel Martinez
Miguel Martinez
Anthony Ross
Juan Soto
Miguel Martinez
Miguel Martinez
John Ross
Juan Martinez
Juan Martinez
José Soto
José Soto
Miguel Rodríguez
David Ross
John Ross
José Soto
Juan Martinez
Juan Soto
Juan Soto


Here, we have swapped our majority-American default profile with a majority-Dominican profile instead.

I have now migrated my workflow to a local instance. Let's check to make sure it works:

In [9]:
import sys
sys.path.append("../src")

from generator import *

In [10]:
profiles = load_profiles("../data/profiles.json")

print(generate_batch(profiles, 10, mode="mixed"))

John Parker
José Soto
John Parker
David Parker
John Ross
David Ross
Miguel Martinez
David Ross
David Ross
Anthony Parker
